In [3]:
#Andrés Pabón - Parcial 3
import csv
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

usos = ["Medicinal", "Alimenticio", "Maderable"]

archivos = {
    "brigadas": "brigadas.csv",
    "conglomerados": "conglomerados.csv",
    "inspecciones": "inspecciones.csv",
    "conteos": "conteos.csv",
    "muestras": "muestras.csv"
}

campos = {
    "brigadas": ["brigada", "lider", "profesionales", "coinvestigadores"],
    "conglomerados": ["conglomerado", "departamento", "municipio", "latitud", "longitud", "brigada"],
    "inspecciones": ["inspeccion", "conglomerado"],
    "conteos": ["inspeccion", "subparcela", "fustales", "latizales", "brinzales"],
    "muestras": ["codigo", "inspeccion", "nombreComun", "uso", "nombreCientifico"]
}

def leerArchivo(archivo):
    try:
        with open(archivo, "r", newline="", encoding="utf-8") as file:
            return list(csv.DictReader(file))
    except FileNotFoundError:
        return []

def guardarArchivo(nombre, lista):
    with open(archivos[nombre], "w", newline="", encoding="utf-8") as file:
        escritor = csv.DictWriter(file, fieldnames=campos[nombre])
        escritor.writeheader()
        escritor.writerows(lista)

def cargarDatos():
    global brigadas, conglomerados, inspecciones, conteos, muestras

    brigadas = leerArchivo(archivos["brigadas"])
    conglomerados = leerArchivo(archivos["conglomerados"])
    inspecciones = leerArchivo(archivos["inspecciones"])
    conteos = leerArchivo(archivos["conteos"])
    muestras = leerArchivo(archivos["muestras"])

cargarDatos()

if not any(os.path.exists(archivo) for archivo in archivos.values()):
    brigadas = [
        {"brigada": "B001", "lider": "Jair Sánchez", "profesionales": "Ana Rojas (botánica), Luis Mena (ingeniero forestal)", "coinvestigadores": "Ever Blandón, Marta Palacios"},
        {"brigada": "B002", "lider": "Camila Duarte", "profesionales": "Pedro Ortiz (biólogo), Sofía Vargas (ecóloga)", "coinvestigadores": "Hernando Rueda"}
    ]

    conglomerados = [
        {"conglomerado": "C001", "departamento": "Chocó", "municipio": "Quibdó", "latitud": 5.6947, "longitud": -76.6583, "brigada": "B001"},
        {"conglomerado": "C002", "departamento": "Santander", "municipio": "San Vicente de Chucurí", "latitud": 6.8820, "longitud": -73.4120, "brigada": "B002"},
        {"conglomerado": "C003", "departamento": "Chocó", "municipio": "Istmina", "latitud": 5.1519, "longitud": -76.6847, "brigada": "B001"}
    ]

    inspecciones = [
        {"inspeccion": "I001", "conglomerado": "C001"},
        {"inspeccion": "I002", "conglomerado": "C002"},
        {"inspeccion": "I003", "conglomerado": "C003"}
    ]

    conteos = [
        {"inspeccion": "I001", "subparcela": 1, "fustales": 12, "latizales": 25, "brinzales": 40},
        {"inspeccion": "I001", "subparcela": 2, "fustales": 9, "latizales": 18, "brinzales": 35},
        {"inspeccion": "I002", "subparcela": 1, "fustales": 8, "latizales": 15, "brinzales": 27},
        {"inspeccion": "I003", "subparcela": 1, "fustales": 10, "latizales": 21, "brinzales": 33}
    ]

    muestras = [
        {"codigo": "M001", "inspeccion": "I001", "nombreComun": "Cedro", "uso": "Maderable", "nombreCientifico": "Cedrela odorata"},
        {"codigo": "M002", "inspeccion": "I001", "nombreComun": "Uña de gato", "uso": "Medicinal", "nombreCientifico": "Uncaria tomentosa"},
        {"codigo": "M003", "inspeccion": "I002", "nombreComun": "Caimito", "uso": "Alimenticio", "nombreCientifico": ""},
        {"codigo": "M004", "inspeccion": "I003", "nombreComun": "Guadua", "uso": "Maderable", "nombreCientifico": "Guadua angustifolia"}
    ]

    for nombre, lista in [
        ("brigadas", brigadas),
        ("conglomerados", conglomerados),
        ("inspecciones", inspecciones),
        ("conteos", conteos),
        ("muestras", muestras)
    ]:
        guardarArchivo(nombre, lista)

def buscar(lista, campo, valor):
    return next(
        (fila for fila in lista if str(fila.get(campo, "")) == str(valor)),
        None
    )

def crearCodigo(letra, lista, campo):
    numeros = [
        int(fila[campo][1:])
        for fila in lista
        if fila.get(campo, "").startswith(letra)
        and fila[campo][1:].isdigit()
    ]
    return letra + str(max(numeros, default=0) + 1).zfill(3)

def listarBrigadas():
    return [fila["brigada"] for fila in brigadas]

def listarConglomerados():
    return [fila["conglomerado"] for fila in conglomerados]

def listarInspecciones():
    return [fila["inspeccion"] for fila in inspecciones]

def registrarBrigada(lider, profesionales, coinvestigadores):
    codigo = crearCodigo("B", brigadas, "brigada")
    brigadas.append({
        "brigada": codigo,
        "lider": lider,
        "profesionales": profesionales,
        "coinvestigadores": coinvestigadores
    })
    guardarArchivo("brigadas", brigadas)
    return f"Brigada {codigo} registrada."

def registrarConglomerado(departamento, municipio, latitud, longitud, brigada):
    codigo = crearCodigo("C", conglomerados, "conglomerado")
    conglomerados.append({
        "conglomerado": codigo,
        "departamento": departamento,
        "municipio": municipio,
        "latitud": latitud,
        "longitud": longitud,
        "brigada": brigada
    })
    guardarArchivo("conglomerados", conglomerados)
    return f"Conglomerado {codigo} registrado."

def registrarInspeccion(conglomerado):
    codigo = crearCodigo("I", inspecciones, "inspeccion")
    inspecciones.append({
        "inspeccion": codigo,
        "conglomerado": conglomerado
    })
    guardarArchivo("inspecciones", inspecciones)
    return f"Inspección {codigo} registrada."

def registrarConteo(inspeccion, subparcela, fustales, latizales, brinzales):
    conteos.append({
        "inspeccion": inspeccion,
        "subparcela": subparcela,
        "fustales": fustales,
        "latizales": latizales,
        "brinzales": brinzales
    })
    guardarArchivo("conteos", conteos)
    return "Conteo registrado."

def registrarMuestra(inspeccion, nombreComun, uso, nombreCientifico):
    codigo = crearCodigo("M", muestras, "codigo")
    muestras.append({
        "codigo": codigo,
        "inspeccion": inspeccion,
        "nombreComun": nombreComun,
        "uso": uso,
        "nombreCientifico": nombreCientifico
    })
    guardarArchivo("muestras", muestras)
    return f"Muestra {codigo} registrada."

def consultarInspeccion(codigo):
    inspeccion = buscar(inspecciones, "inspeccion", codigo)

    if not inspeccion:
        return None

    conglomerado = buscar(
        conglomerados, "conglomerado", inspeccion["conglomerado"]
    )

    if not conglomerado:
        return None

    return {
        "conglomerado": conglomerado,
        "brigada": buscar(brigadas, "brigada", conglomerado["brigada"]),
        "conteos": [
            fila for fila in conteos
            if fila["inspeccion"] == codigo
        ],
        "muestras": [
            fila for fila in muestras
            if fila["inspeccion"] == codigo
        ]
    }

salida = widgets.Output()
contenedor = widgets.VBox()

listaPrincipal = widgets.Dropdown(description="Seleccionar:", options=[])
listaSecundaria = widgets.Dropdown(description="Uso:", options=usos)

textoUno = widgets.Text()
textoDos = widgets.Text()
textoTres = widgets.Text()

numeroUno = widgets.FloatText()
numeroDos = widgets.FloatText()
numeroTres = widgets.FloatText()
numeroCuatro = widgets.FloatText()

botonAccion = widgets.Button(description="Guardar", button_style="success")
botonActualizar = widgets.Button(description="Actualizar inspecciones")

opcionActual = ""

def actualizarInspecciones(boton=None):
    seleccion = listaPrincipal.value

    inspecciones[:] = leerArchivo(archivos["inspecciones"])
    opciones = listarInspecciones()
    listaPrincipal.options = opciones

    if seleccion in opciones:
        listaPrincipal.value = seleccion
    elif opciones:
        listaPrincipal.value = opciones[-1]

    with salida:
        clear_output()
        if opciones:
            print("Lista de inspecciones actualizada.")
        else:
            print("No hay inspecciones registradas.")

def clicMenu(boton):
    global opcionActual

    opcionActual = boton.description
    botonAccion.description = (
        "Consultar" if opcionActual == "Consultar inspección" else "Guardar"
    )

    textoUno.value = ""
    textoDos.value = ""
    textoTres.value = ""
    numeroUno.value = 0
    numeroDos.value = 0
    numeroTres.value = 0
    numeroCuatro.value = 0

    with salida:
        clear_output()

    if opcionActual == "Registrar brigada":
        textoUno.description = "Líder:"
        textoDos.description = "Profesionales:"
        textoTres.description = "Coinvestigadores locales:"
        contenedor.children = [
            textoUno, textoDos, textoTres, botonAccion
        ]

    elif opcionActual == "Registrar conglomerado":
        textoUno.description = "Departamento:"
        textoDos.description = "Municipio:"
        numeroUno.description = "Latitud:"
        numeroDos.description = "Longitud:"
        listaPrincipal.description = "Brigada:"
        listaPrincipal.options = listarBrigadas()
        contenedor.children = [
            textoUno, textoDos, numeroUno, numeroDos,
            listaPrincipal, botonAccion
        ]

    elif opcionActual == "Registrar inspección":
        listaPrincipal.description = "Conglomerado:"
        listaPrincipal.options = listarConglomerados()
        contenedor.children = [listaPrincipal, botonAccion]

    elif opcionActual == "Registrar conteo":
        listaPrincipal.description = "Inspección:"
        listaPrincipal.options = listarInspecciones()
        numeroUno.description = "Subparcela:"
        numeroUno.value = 1
        numeroDos.description = "Fustales:"
        numeroTres.description = "Latizales:"
        numeroCuatro.description = "Brinzales:"
        contenedor.children = [
            listaPrincipal, numeroUno, numeroDos,
            numeroTres, numeroCuatro, botonAccion
        ]

    elif opcionActual == "Registrar muestra":
        listaPrincipal.description = "Inspección:"
        listaPrincipal.options = listarInspecciones()
        textoUno.description = "Nombre común:"
        textoDos.description = "Nombre científico:"
        contenedor.children = [
            listaPrincipal, textoUno, listaSecundaria,
            textoDos, botonAccion
        ]

    elif opcionActual == "Consultar inspección":
        actualizarInspecciones()
        listaPrincipal.description = "Inspección:"
        contenedor.children = [
            listaPrincipal, botonActualizar, botonAccion
        ]

def clicAccion(boton):
    opcion = opcionActual

    with salida:
        clear_output()

        try:
            if opcion == "Registrar brigada":
                if not textoUno.value.strip():
                    print("Escribe el nombre del líder.")
                    return

                print(registrarBrigada(
                    textoUno.value.strip(),
                    textoDos.value.strip(),
                    textoTres.value.strip()
                ))

            elif opcion == "Registrar conglomerado":
                if (
                    not textoUno.value.strip()
                    or not textoDos.value.strip()
                    or listaPrincipal.value is None
                ):
                    print("Completa los datos y selecciona una brigada.")
                    return

                print(registrarConglomerado(
                    textoUno.value.strip(),
                    textoDos.value.strip(),
                    numeroUno.value,
                    numeroDos.value,
                    listaPrincipal.value
                ))

            elif opcion == "Registrar inspección":
                if listaPrincipal.value is None:
                    print("Primero registra un conglomerado.")
                    return

                print(registrarInspeccion(listaPrincipal.value))

            elif opcion == "Registrar conteo":
                valores = [
                    numeroUno.value, numeroDos.value,
                    numeroTres.value, numeroCuatro.value
                ]

                if listaPrincipal.value is None:
                    print("Primero registra una inspección.")
                    return

                if any(valor < 0 or not valor.is_integer() for valor in valores):
                    print("La subparcela y los conteos deben ser enteros no negativos.")
                    return

                print(registrarConteo(
                    listaPrincipal.value,
                    *(int(valor) for valor in valores)
                ))

            elif opcion == "Registrar muestra":
                if listaPrincipal.value is None or not textoUno.value.strip():
                    print("Selecciona una inspección y escribe el nombre común.")
                    return

                print(registrarMuestra(
                    listaPrincipal.value,
                    textoUno.value.strip(),
                    listaSecundaria.value,
                    textoDos.value.strip()
                ))

            elif opcion == "Consultar inspección":
                actualizarInspecciones()
                codigo = listaPrincipal.value
                info = consultarInspeccion(codigo) if codigo else None

                if not info:
                    print("No se encontró la inspección o sus datos relacionados.")
                    return

                conglomerado = info["conglomerado"]
                brigada = info["brigada"]

                print("INSPECCIÓN", codigo)
                print()
                print("CONGLOMERADO", conglomerado["conglomerado"])
                print(
                    "Ubicación:",
                    conglomerado["municipio"] + ",",
                    conglomerado["departamento"]
                )
                print(
                    "Centroide:",
                    conglomerado["latitud"],
                    conglomerado["longitud"]
                )

                if brigada:
                    print()
                    print("BRIGADA", brigada["brigada"])
                    print("Líder:", brigada["lider"])
                    print("Profesionales:", brigada["profesionales"])
                    print("Coinvestigadores locales:", brigada["coinvestigadores"])

                print()
                print("CONTEO DE ÁRBOLES")

                if info["conteos"]:
                    for fila in info["conteos"]:
                        print(
                            f"Subparcela {fila['subparcela']} | "
                            f"Fustales: {fila['fustales']} | "
                            f"Latizales: {fila['latizales']} | "
                            f"Brinzales: {fila['brinzales']}"
                        )
                else:
                    print("Sin conteos registrados.")

                print()
                print("MUESTREO")

                if info["muestras"]:
                    for fila in info["muestras"]:
                        cientifico = (
                            fila["nombreCientifico"]
                            or "(pendiente de clasificación)"
                        )
                        print(
                            f"{fila['codigo']} | {fila['nombreComun']} | "
                            f"{fila['uso']} | {cientifico}"
                        )
                else:
                    print("Sin muestras registradas.")

        except (ValueError, OSError, KeyError) as error:
            print("No se pudo completar la operación:", error)

botonesMenu = [
    widgets.Button(description=nombre)
    for nombre in [
        "Registrar brigada",
        "Registrar conglomerado",
        "Registrar inspección",
        "Registrar conteo",
        "Registrar muestra",
        "Consultar inspección"
    ]
]

for boton in botonesMenu:
    boton.on_click(clicMenu)

botonActualizar.on_click(actualizarInspecciones)
botonAccion.on_click(clicAccion)

display(
    widgets.HBox(
        botonesMenu,
        layout=widgets.Layout(flex_flow="row wrap")
    )
)
display(contenedor)
display(salida)

VBox()

Output()